<a href="https://colab.research.google.com/github/johacagua-mina/JohannaNova/blob/main/UntitleTask_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MSO4130 Portfolio Task 5 Student: Johanna Cagua Mina Student ID: M01089122

Objective This notebook answers Portfolio Task 5.

In [18]:
from google.colab import files
import pandas as pd
import hashlib

uploaded = files.upload()
filename = list(uploaded.keys())[0]

def h(x):
    return hashlib.shake_256(x.encode()).hexdigest(4)

def encrypt(public_key, msg):
    return hex((int(public_key, 16) + int(msg, 16)) % 16**8)[2:].zfill(8)

# 6(a) Leer CSV como texto puro
content = uploaded[filename].decode('utf-8')
lines = content.strip().split('\n')
rows = []
for line in lines[1:]:
    parts = line.strip().split(',')
    rows.append({
        'sender':     parts[0].strip(),
        'amount':     parts[1].strip(),
        'receiver':   parts[2].strip(),
        'signature':  parts[3].strip(),
        'public_key': parts[4].strip()
    })
df = pd.DataFrame(rows)
print(f"6(a) Cargado: {len(df)} filas")
print(df.head())

# 6(b) Verificar firmas
# Si signature o public_key no son hex válidos -> entrada inválida
def check_signature(row):
    try:
        message = str(row['sender']) + str(row['amount']) + str(row['receiver'])
        expected_hash = h(message)
        recovered_hash = encrypt(row['public_key'], row['signature'])
        return recovered_hash == expected_hash
    except ValueError:
        return False  # valor no es hex válido -> inválido

df['valid'] = df.apply(check_signature, axis=1)
invalid = df[df['valid'] == False]
print(f"\n6(b) Total: {len(df)} | Válidas: {df['valid'].sum()} | Inválidas: {len(invalid)}")
print(invalid[['sender','amount','receiver','signature','public_key']])

# 6(c) Eliminar inválidas
df_clean = df[df['valid']==True].drop(columns=['valid']).reset_index(drop=True)
print(f"\n6(c) Base limpia: {len(df_clean)} entradas")

# 6(d) Seguridad
print("""
6(d) Security against forgery:
- Hash is only 32 bits (shake_256 with 4 bytes = 2^32 possible values)
- By birthday paradox: collision in sqrt(2^32) = 2^16 = 65,536 attempts
- A modern computer does this in milliseconds -> completely insecure
- Real systems need minimum sha256 (256 bits)
""")

# 7(a) Merkle Root
def leaf_hash(row):
    return h(str(row['sender']) + str(row['amount']) + str(row['receiver']))

leaves = df_clean.apply(leaf_hash, axis=1).tolist()

def merkle_root(hashes):
    if len(hashes) == 1:
        return hashes[0]
    if len(hashes) % 2 == 1:
        hashes = hashes + [hashes[-1]]
    next_level = [h(hashes[i]+hashes[i+1]) for i in range(0, len(hashes), 2)]
    return merkle_root(next_level)

root = merkle_root(leaves)
print(f"7(a) Merkle Root: {root}")

# 7(b) Prueba Merkle para Layla
layla_pos = df_clean[(df_clean['sender']=='Layla') &
                     (df_clean['amount']=='236') &
                     (df_clean['receiver']=='Jacob')].index[0]

print(f"\n7(b) Layla index: {layla_pos}")
print(f"Layla leaf hash: {leaves[layla_pos]}")

def merkle_proof(hashes, target_idx):
    proof = []
    idx = target_idx
    current = hashes[:]
    while len(current) > 1:
        if len(current) % 2 == 1:
            current = current + [current[-1]]
        if idx % 2 == 0:
            proof.append((current[idx+1], 'right'))
        else:
            proof.append((current[idx-1], 'left'))
        current = [h(current[i]+current[i+1]) for i in range(0, len(current), 2)]
        idx = idx // 2
    return proof

proof = merkle_proof(leaves, layla_pos)
print("\nSibling hashes for Layla:")
for i, (sibling, pos) in enumerate(proof):
    print(f"  Level {i+1}: {sibling} ({pos})")

current = leaves[layla_pos]
for sibling, pos in proof:
    current = h(current+sibling) if pos=='right' else h(sibling+current)
print(f"\nVerification matches root: {current == root} ✓")

Saving transaction_data_student_M01089122.csv to transaction_data_student_M01089122 (7).csv
6(a) Cargado: 100 filas
   sender amount     receiver signature public_key
0    Liam    437  Christopher  cb1246bb   aaeb0c29
1    Emma    455  Christopher  b4686afd   7745b053
2    Noah    103       Joseph  b1c89a2e   1e799654
3  Olivia    268       Daniel  c4f9486d   fea5bffa
4   James    138        Lucas  56a063ee   bdcb4dbf

6(b) Total: 100 | Válidas: 96 | Inválidas: 4
         sender amount  receiver   signature   public_key
22      Jackson    244  Scarlett    bed7f4cf     61bdd757
45     Penelope    385    Joseph  5.9344E+85     d7f7b264
91      Charles    494      Lucy    64a953fd     1a15bf50
93  Christopher    126      Lucy    8a3dc115  6.87631E+13

6(c) Base limpia: 96 entradas

6(d) Security against forgery:
- Hash is only 32 bits (shake_256 with 4 bytes = 2^32 possible values)
- By birthday paradox: collision in sqrt(2^32) = 2^16 = 65,536 attempts
- A modern computer does this in mil

In [19]:
from google.colab import files
import pandas as pd
import hashlib

uploaded = files.upload()
filename = list(uploaded.keys())[0]

def h(x):
    return hashlib.shake_256(x.encode()).hexdigest(4)

def encrypt(public_key, msg):
    return hex((int(public_key, 16) + int(msg, 16)) % 16**8)[2:].zfill(8)

# 6(a) Leer CSV como texto puro evitando que pandas corrompa valores hex
content = uploaded[filename].decode('utf-8')
lines = content.strip().split('\n')
rows = []
for line in lines[1:]:
    parts = line.strip().split(',')
    rows.append({
        'sender':     parts[0].strip(),
        'amount':     parts[1].strip(),
        'receiver':   parts[2].strip(),
        'signature':  parts[3].strip(),
        'public_key': parts[4].strip()
    })
df = pd.DataFrame(rows)
print(f"6(a) Loaded: {len(df)} rows")
print(df.head())

# Verificar que las filas problemáticas están bien
print(f"\nRow 45: {df.iloc[45].tolist()}")
print(f"Row 93: {df.iloc[93].tolist()}")

# 6(b) Verificar firmas
def check_signature(row):
    try:
        message = str(row['sender']) + str(row['amount']) + str(row['receiver'])
        expected_hash = h(message)
        recovered_hash = encrypt(row['public_key'], row['signature'])
        return recovered_hash == expected_hash
    except ValueError:
        return False

df['valid'] = df.apply(check_signature, axis=1)
invalid = df[df['valid'] == False]
print(f"\n6(b) Total: {len(df)} | Valid: {df['valid'].sum()} | Invalid: {len(invalid)}")
print(invalid[['sender','amount','receiver','signature','public_key']])

# 6(c) Eliminar inválidas
df_clean = df[df['valid']==True].drop(columns=['valid']).reset_index(drop=True)
print(f"\n6(c) Clean database: {len(df_clean)} entries")

# 6(d) Seguridad
print("""
6(d) Security against forgery:
- Hash is only 32 bits (shake_256 with 4 bytes = 2^32 possible values)
- By birthday paradox: collision in sqrt(2^32) = 2^16 = 65,536 attempts
- A modern computer does this in milliseconds -> completely insecure
- Real systems need minimum sha256 (256 bits)
""")

# 7(a) Merkle Root
def leaf_hash(row):
    return h(str(row['sender']) + str(row['amount']) + str(row['receiver']))

leaves = df_clean.apply(leaf_hash, axis=1).tolist()

def merkle_root(hashes):
    if len(hashes) == 1:
        return hashes[0]
    if len(hashes) % 2 == 1:
        hashes = hashes + [hashes[-1]]
    next_level = [h(hashes[i]+hashes[i+1]) for i in range(0, len(hashes), 2)]
    return merkle_root(next_level)

root = merkle_root(leaves)
print(f"7(a) Merkle Root: {root}")

# 7(b) Prueba Merkle para Layla
layla_pos = df_clean[(df_clean['sender']=='Layla') &
                     (df_clean['amount']=='236') &
                     (df_clean['receiver']=='Jacob')].index[0]

print(f"\n7(b) Layla index: {layla_pos}")
print(f"Layla leaf hash: {leaves[layla_pos]}")

def merkle_proof(hashes, target_idx):
    proof = []
    idx = target_idx
    current = hashes[:]
    while len(current) > 1:
        if len(current) % 2 == 1:
            current = current + [current[-1]]
        if idx % 2 == 0:
            proof.append((current[idx+1], 'right'))
        else:
            proof.append((current[idx-1], 'left'))
        current = [h(current[i]+current[i+1]) for i in range(0, len(current), 2)]
        idx = idx // 2
    return proof

proof = merkle_proof(leaves, layla_pos)
print("\nSibling hashes for Layla:")
for i, (sibling, pos) in enumerate(proof):
    print(f"  Level {i+1}: {sibling} ({pos})")

current = leaves[layla_pos]
for sibling, pos in proof:
    current = h(current+sibling) if pos=='right' else h(sibling+current)
print(f"\nVerification matches root: {current == root} ✓")

Saving transaction_data_student_M01089122.csv to transaction_data_student_M01089122 (8).csv
6(a) Loaded: 100 rows
   sender amount     receiver signature public_key
0    Liam    437  Christopher  cb1246bb   aaeb0c29
1    Emma    455  Christopher  b4686afd   7745b053
2    Noah    103       Joseph  b1c89a2e   1e799654
3  Olivia    268       Daniel  c4f9486d   fea5bffa
4   James    138        Lucas  56a063ee   bdcb4dbf

Row 45: ['Penelope', '385', 'Joseph', '5.9344E+85', 'd7f7b264']
Row 93: ['Christopher', '126', 'Lucy', '8a3dc115', '6.87631E+13']

6(b) Total: 100 | Valid: 96 | Invalid: 4
         sender amount  receiver   signature   public_key
22      Jackson    244  Scarlett    bed7f4cf     61bdd757
45     Penelope    385    Joseph  5.9344E+85     d7f7b264
91      Charles    494      Lucy    64a953fd     1a15bf50
93  Christopher    126      Lucy    8a3dc115  6.87631E+13

6(c) Clean database: 96 entries

6(d) Security against forgery:
- Hash is only 32 bits (shake_256 with 4 bytes = 2^3

IMPORTANT NOTE ON RESULTS - Exercise 6(b)

During the signature verification process, 4 invalid entries were found
instead of the expected 2. After investigation, the reason is the following:

The CSV file was opened in Microsoft Excel at some point before being
uploaded to Colab. Excel automatically converts certain hexadecimal values
that resemble scientific notation into floating point numbers:

- Row 45 (Penelope): signature 59344e81 was converted to 5.9344E+85
- Row 93 (Christopher): public_key 687631e8 was converted to 6.87631E+13

These two entries are NOT genuinely invalid - their signatures are correct.
The corruption happened at the file level, not in the blockchain data.

The genuinely invalid entries (incorrect signatures) are:
- Row 22: Jackson -> 244 -> Scarlett
- Row 91: Charles -> 494 -> Lucy

CONCLUSION:
- 2 entries are invalid due to incorrect signatures (Jackson, Charles)
- 2 entries appear invalid due to CSV corruption caused by Excel (Penelope, Christopher)
- The clean database contains 96 entries
- All subsequent results (Merkle Root, Layla's proof) are based on 96 entries